In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
from pathlib import Path

# Log-Verzeichnis
LOG_DIR = Path(cfg.paths.logs_dir)

# Verzeichnis anlegen, wenn nicht vorhanden
LOG_DIR.mkdir(exist_ok=True)

print(f"Log-Verzeichnis: {LOG_DIR.resolve()}")

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime

timing_results = []

def run_notebook(path):
    nb_path = Path(path)
    nb_name = nb_path.stem
    log_file = LOG_DIR / f"{nb_name}.log"

    print(f"\nStarte: {path}")
    print(f"Logfile: {log_file.name}")

    cmd = [
        "papermill",
        str(nb_path),
        str(nb_path),     # inplace
        "--log-output"
    ]

    start_time = datetime.now()

    with open(log_file, "a", encoding="utf-8") as lf:
        lf.write(f"\n=== {start_time} | START {path} ===\n")

        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            print(line, end="")
            lf.write(line)

        exit_code = process.wait()
        end_time = datetime.now()

        if exit_code != 0:
            raise RuntimeError(f"Fehler in {path}")

        lf.write(f"=== {end_time} | END {path} ===\n")

    elapsed = end_time - start_time
    timing_results.append({
        "notebook": nb_name,
        "start": start_time.strftime("%H:%M:%S"),
        "end": end_time.strftime("%H:%M:%S"),
        "duration_seconds": round(elapsed.total_seconds(), 1)
    })
    print(f"Erledigt: {path} ({elapsed.total_seconds():.1f}s)")

In [ ]:
pipeline = cfg.pipeline.notebooks

# Alle Notebooks BIS 99_statistics_md ausführen
for nb in pipeline:
    if nb == "99_statistics_md.ipynb":
        break
    run_notebook(nb)

# --- Timing persistieren (BEVOR 99 läuft) ---
total_seconds = sum(r["duration_seconds"] for r in timing_results)
total_min = int(total_seconds // 60)
total_sec = round(total_seconds % 60, 1)

lines = []
lines.append("| Notebook | Start | Ende | Dauer (s) |")
lines.append("|----------|-------|------|-----------|")
for r in timing_results:
    lines.append(f"| {r['notebook']} | {r['start']} | {r['end']} | {r['duration_seconds']} |")
lines.append(f"| **Gesamt** | | | **{total_seconds:.1f}** ({total_min}m {total_sec}s) |")

timing_md = "\n".join(lines)
timing_path = cfg.asset_path("pipeline_timing")
with open(timing_path, "w", encoding="utf-8") as f:
    f.write(timing_md)

print(f"Pipeline-Timing gespeichert: {timing_path}")

# --- Jetzt 99_statistics_md ausführen ---
run_notebook("99_statistics_md.ipynb")

print("\nGesamte Pipeline erfolgreich durchlaufen.")